In [1]:
import os

base_path = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
# 최상위에 뭐가 있는지 확인
print(os.listdir(base_path))

['test_series.csv', 'train_series.csv', 'sample_submission.csv', 'test_series', 'train_series', 'train.csv', 'test.csv']


In [2]:
import pandas as pd

train_df = pd.read_csv(f'{base_path}/train.csv')  
print(train_df.shape)
train_df.head()

(4407, 14)


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pydicom
import matplotlib.pyplot as plt

# 첫 번째 스터디의 폴더 구조를 확인
study_folders = os.listdir(f'{base_path}/train_series')
print(f"총 스터디 수: {len(study_folders)}")
print(f"첫 스터디: {study_folders[0]}")

# 그 스터디 안의 파일들 확인
first_study_path = f'{base_path}/train_series/{study_folders[0]}'
print(os.listdir(first_study_path))

총 스터디 수: 4407
첫 스터디: 1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839
['1.2.826.0.1.3680043.8.498.28798546215369442215420222435134673142', '1.2.826.0.1.3680043.8.498.77574746339062590058612008891367008722', '1.2.826.0.1.3680043.8.498.11095855317733507937819924915747155492', '1.2.826.0.1.3680043.8.498.20777921698448069425369525866577598114', '1.2.826.0.1.3680043.8.498.12352363451108476518578557153043561806']


In [4]:
from tqdm import tqdm

def scan_studies(study_list):
    records = []
    for study_id in tqdm(study_list):
        study_path = f'{base_path}/train_series/{study_id}'
        for series_id in os.listdir(study_path):
            series_path = f'{study_path}/{series_id}'
            dicom_files = os.listdir(series_path)
            if not dicom_files:
                continue
            sample_file = f'{series_path}/{dicom_files[0]}'
            try:
                dcm = pydicom.dcmread(sample_file, stop_before_pixels=True)
                records.append({
                    'study_id': study_id,
                    'series_id': series_id,
                    'num_slices': len(dicom_files),
                    'series_description': getattr(dcm, 'SeriesDescription', None),
                    'rows': getattr(dcm, 'Rows', None),
                    'columns': getattr(dcm, 'Columns', None),
                    'slice_thickness': getattr(dcm, 'SliceThickness', None),
                })
            except Exception as e:
                print(f"에러 ({sample_file}): {e}")
    return records

import math

chunk_size = 1000
total = len(study_folders)
num_chunks = math.ceil(total / chunk_size)  # 4407 / 1000 → 올림해서 5

print(f"전체 {total}개를 {num_chunks}개 구간으로 나눠서 처리해요")

for chunk_index in range(num_chunks):
    start = chunk_index * chunk_size
    end = start + chunk_size
    chunk = study_folders[start:end]

    print(f"[{chunk_index+1}/{num_chunks}] {start}번째 ~ {min(end, total)}번째 처리 중")
    records = scan_studies(chunk)

    df = pd.DataFrame(records)
    df.to_csv(f'metadata_chunk_{chunk_index}.csv', index=False)

전체 4407개를 5개 구간으로 나눠서 처리해요
[1/5] 0번째 ~ 1000번째 처리 중


100%|██████████| 1000/1000 [01:34<00:00, 10.53it/s]


[2/5] 1000번째 ~ 2000번째 처리 중


100%|██████████| 1000/1000 [01:29<00:00, 11.15it/s]


[3/5] 2000번째 ~ 3000번째 처리 중


100%|██████████| 1000/1000 [01:28<00:00, 11.36it/s]


[4/5] 3000번째 ~ 4000번째 처리 중


100%|██████████| 1000/1000 [01:40<00:00,  9.99it/s]


[5/5] 4000번째 ~ 4407번째 처리 중


100%|██████████| 407/407 [00:41<00:00,  9.75it/s]


In [5]:
# 파일 합치기 

import glob

all_files = sorted(glob.glob('metadata_chunk_*.csv'))
print(f"찾은 파일들: {all_files}")

combined_df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
combined_df.to_csv('metadata_scan_full.csv', index=False)
print(f"전체 행 수: {len(combined_df)}")

찾은 파일들: ['metadata_chunk_0.csv', 'metadata_chunk_1.csv', 'metadata_chunk_2.csv', 'metadata_chunk_3.csv', 'metadata_chunk_4.csv']
전체 행 수: 24371
